In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import librosa
import numpy as np
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

from torch.utils.data import TensorDataset, DataLoader
import wandb

wandb.init(
    project="23f2000441-t12026",
    name="Milestone-4-CRNN",
    mode="offline"   # Kaggle ke liye best
)

In [ ]:
def extract_spectrogram(file_path):
    y, sr = librosa.load(file_path, duration=5)   # 🔥 fast
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    mel_db = mel_db[:128, :128]
    
    mean = np.mean(mel_db)
    std = np.std(mel_db)
    
    if std != 0:
        mel_db = (mel_db - mean) / std
    
    return mel_db

In [ ]:
base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

X = []
y = []

stems = ["vocals.wav", "drums.wav", "bass.wav", "others.wav"]

for genre in os.listdir(base_path):
    genre_path = os.path.join(base_path, genre)
    
    for song in tqdm(os.listdir(genre_path)[:100]):   # keep small first
        
        for stem in stems:
            file_path = os.path.join(genre_path, song, stem)
            
            try:
                spec = extract_spectrogram(file_path)
                X.append(spec)
                y.append(genre)
            except:
                continue

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc, test_size=0.2, random_state=42
)

In [ ]:
X_train = torch.tensor(X_train).unsqueeze(1).float()
X_val = torch.tensor(X_val).unsqueeze(1).float()

y_train = torch.tensor(y_train).long()
y_val = torch.tensor(y_val).long()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

In [ ]:
class CRNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # CNN part
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # LSTM part
        self.lstm = nn.LSTM(
            input_size=64 * 16,   # features
            hidden_size=128,
            batch_first=True
        )
        
        # FC
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.cnn(x)  
        # shape: (batch, 64, 16, 16)

        x = x.permute(0, 3, 1, 2)  
        # (batch, time=16, channels=64, height=16)

        x = x.reshape(x.size(0), x.size(1), -1)  
        # (batch, time=16, features=64*16)

        lstm_out, _ = self.lstm(x)

        out = lstm_out[:, -1, :]  # last time step

        out = self.fc(out)

        return out

In [ ]:
model = CRNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

epochs = 30

for epoch in range(epochs):
    
    model.train()
    total_loss = 0
    
    for xb, yb in train_loader:
        
        outputs = model(xb)
        loss = criterion(outputs, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss}")
    
    # 🔥 wandb log (ONLY ONCE, correct indentation)
    wandb.log({
        "epoch": epoch+1,
        "loss": total_loss
    })

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        preds = torch.argmax(outputs, axis=1)
        
        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

f1 = f1_score(all_labels, all_preds, average="macro")

print("🔥 CRNN Macro F1:", f1)
wandb.log({"F1_score": f1})
wandb.finish()